<a href="https://colab.research.google.com/github/serin19538-jpg/MS49900_ADV/blob/main/Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
## [별도 노트북] 용매 38종의 온도 의존 물성을 thermo 데이터베이스에서 추출 → dict로 출력
## ※ 대회 노트북에는 설치하지 말 것 (numpy/pandas 버전이 바뀔 수 있음)
!pip install -q thermo
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from thermo import Chemical

CAS = {  ## 키 = 대회 데이터의 용매 열 이름
    "EC": "96-49-1", "PC": "108-32-7", "DMC": "616-38-6", "EMC": "623-53-0", "DEC": "105-58-8",
    "DME": "110-71-4", "DMSO": "67-68-5", "AN": "75-05-8", "MOEMC": "35466-86-6",   ## MOEMC CAS 확인 필요
    "TFP": "358-63-4", "EA": "141-78-6", "MA": "79-20-9", "FEC": "114435-02-8", "DOL": "646-06-0",
    "2-MeTHF": "96-47-9", "DMM": "109-87-5", "Freon 11": "75-69-4", "Methylene chloride": "75-09-2",
    "THF": "109-99-9", "Toluene": "108-88-3", "Sulfolane": "126-33-0", "2-Glyme": "111-96-6",
    "3-Glyme": "112-49-2", "4-Glyme": "143-24-8", "3-Me-2-Oxazolidinone": "19836-78-3",
    "3-MeSulfolane": "872-93-5", "Ethyldiglyme": "112-36-7", "DMF": "68-12-2", "Ethylbenzene": "100-41-4",
    "Ethylmonoglyme": "629-14-1", "Benzene": "71-43-2", "g-Butyrolactone": "96-48-0", "Cumene": "98-82-8",
    "Propylsulfone": "598-03-8", "Pseudocumeme": "95-63-6", "TEOS": "78-10-4", "m-Xylene": "108-38-3",
    "o-Xylene": "95-47-6",
}
B_NOW = {  ## 현재 C-14의 ANDRADE_B (비교용)
    "EC": 1800, "PC": 1900, "DMC": 1000, "EMC": 1100, "DEC": 1200, "DME": 950, "DMSO": 1350, "AN": 850,
    "MOEMC": 1300, "TFP": 1800, "EA": 950, "MA": 850, "FEC": 1900, "DOL": 1000, "2-MeTHF": 950, "DMM": 800,
    "Freon 11": 800, "Methylene chloride": 820, "THF": 850, "Toluene": 1050, "Sulfolane": 2200,
    "2-Glyme": 1200, "3-Glyme": 1550, "4-Glyme": 1900, "3-Me-2-Oxazolidinone": 1700, "3-MeSulfolane": 2300,
    "Ethyldiglyme": 1350, "DMF": 900, "Ethylbenzene": 1100, "Ethylmonoglyme": 1000, "Benzene": 1200,
    "g-Butyrolactone": 1400, "Cumene": 1200, "Propylsulfone": 1900, "Pseudocumeme": 1250, "TEOS": 1250,
    "m-Xylene": 1050, "o-Xylene": 1150,
}
T_GRID = np.arange(213.15, 393.16, 10.0)          ## 대회 데이터의 주된 온도 범위

rows, B_LIT, EPS_A, EPS_B, RI_LIT = {}, {}, {}, {}, {}
for s, cas in CAS.items():
    try:
        c0 = Chemical(cas)
    except Exception as e:
        rows[s] = {"name": f"찾지 못함 ({type(e).__name__})"}; continue
    Tm = c0.Tm if c0.Tm else 150.0
    Tb = c0.Tb if c0.Tb else 500.0
    eta, Tv, eps, Te = [], [], [], []
    for T in T_GRID[(T_GRID > Tm + 2) & (T_GRID < Tb - 5)]:   ## 1 atm 액체 구간만
        try:
            c = Chemical(cas, T=T)
            if c.mul: eta.append(c.mul * 1000); Tv.append(T)            ## Pa·s → cP
            if c.permittivity: eps.append(c.permittivity); Te.append(T)
        except Exception:
            pass
    r = {"name": c0.name, "Tm": Tm, "Tb": Tb, "n_eta": len(eta), "n_eps": len(eps), "RI": c0.RI}
    if len(eta) >= 3:                                  ## Andrade: ln η = A + B/T
        x = 1 / np.array(Tv); yv = np.log(eta)
        Bf, Af = np.polyfit(x, yv, 1)
        r.update(B_lit=Bf, B_now=B_NOW[s], eta25_lit=np.exp(Af + Bf / 298.15), r2_eta=np.corrcoef(x, yv)[0, 1] ** 2)
        B_LIT[s] = int(round(Bf))
    if len(eps) >= 3:                                  ## ε(T) = a + b·(298.15/T)  (C-14와 같은 형태)
        b, a = np.polyfit(298.15 / np.array(Te), np.array(eps), 1)
        r.update(eps_a=a, eps_b=b, eps25_lit=a + b)
        EPS_A[s] = round(float(a), 3); EPS_B[s] = round(float(b), 3)
    if c0.RI: RI_LIT[s] = round(float(c0.RI), 4)
    rows[s] = r

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(pd.DataFrame(rows).T.to_string())
print("\n점도 없음/부족:", [s for s in CAS if s not in B_LIT])
print("유전율 없음/부족:", [s for s in CAS if s not in EPS_A])
print("\n## ▼ 아래 네 줄을 복사해서 대회 노트북에 붙여넣기")
print("B_LIT =", B_LIT)
print("EPS_A_LIT =", EPS_A)
print("EPS_B_LIT =", EPS_B)
print("RI_LIT =", RI_LIT)

                                            name      Tm          Tb n_eta n_eps      RI        B_lit B_now  eta25_lit    r2_eta      eps_a       eps_b  eps25_lit
EC                            ethylene carbonate   310.9      519.15     9     9  1.4148   389.769565  1800   0.771832  0.993254 -57.082408  155.184219  98.101811
PC                           propylene carbonate  224.15      514.75    17    17  1.4189   290.746629  1900   0.584253  0.969252  -8.646757   72.174221  63.527464
DMC                           dimethyl carbonate  276.15   363.25613     8     8  1.3687  1078.536618  1000   0.578419  0.999724      3.087        -0.0      3.087
EMC                       ethyl methyl carbonate  259.15      380.65    12    12  1.3778   474.563426  1100   0.409642  0.982139      2.985         0.0      2.985
DEC                            diethyl carbonate  230.15      399.05    17    17  1.3845   1137.91903  1200   0.796125  0.993852       2.82         0.0       2.82
DME                   

In [6]:
## [E-1] 탐색 노트북 설정: 데이터·분할·구간 (대회 노트북과 같은 GroupKFold 분할)
!pip install -q rdkit lightgbm
!git clone -q https://github.com/kwongibaek/MS49900-AI4M.git 2>/dev/null || true
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold

train = pd.read_csv("MS49900-AI4M/Data/train_set.csv")
CATEGORICAL = ["salt", "c units", "solvent ratio type"]
NUMERIC = [c for c in train.columns if c not in CATEGORICAL + ["k"]]
SOLVENTS = [c for c in NUMERIC if c not in ("T", "c")]
X, y = train.drop(columns="k"), train["k"]
groups = train[["salt", "c", "c units", "solvent ratio type"] + SOLVENTS].astype(str).agg("|".join, axis=1)
SPLITS = list(GroupKFold(n_splits=5).split(X, y, groups=groups))
LY = np.log1p(y.values); IS_KG = (X["c units"] == "mol/kg").values
SYS = pd.factorize(X[["salt", "c units", "solvent ratio type"] + SOLVENTS].astype(str).agg("|".join, axis=1))[0]
unseen = np.zeros(len(y), bool)
for fi, vi in SPLITS: unseen[vi] = ~np.isin(SYS[vi], np.unique(SYS[fi]))
SEG = {"mol/kg": IS_KG, "mol/l seen": ~IS_KG & ~unseen, "mol/l unseen": ~IS_KG & unseen}
print({g: int(m.sum()) for g, m in SEG.items()}, "← 대회 노트북 C-16과 같아야 함 (mol/l unseen 1230)")

{'mol/kg': 8491, 'mol/l seen': 1081, 'mol/l unseen': 1238} ← 대회 노트북 C-16과 같아야 함 (mol/l unseen 1230)


In [7]:
## [E-2] 물질 물성 표 — RDKit(구조에서 계산, 전 물질 일관) + thermo(실측/추정 방법 표시)
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors, AllChem
from thermo import Chemical

SOLV_SMILES = {
    "EC": "C1COC(=O)O1", "PC": "CC1COC(=O)O1", "DMC": "COC(=O)OC", "EMC": "CCOC(=O)OC", "DEC": "CCOC(=O)OCC",
    "DME": "COCCOC", "DMSO": "CS(C)=O", "AN": "CC#N", "MOEMC": "COCCOC(=O)OC",
    "TFP": "FC(F)(F)COP(=O)(OCC(F)(F)F)OCC(F)(F)F", "EA": "CCOC(C)=O", "MA": "COC(C)=O", "FEC": "FC1COC(=O)O1",
    "DOL": "C1COCO1", "2-MeTHF": "CC1CCCO1", "DMM": "COCOC", "Freon 11": "FC(Cl)(Cl)Cl", "Methylene chloride": "ClCCl",
    "THF": "C1CCOC1", "Toluene": "Cc1ccccc1", "Sulfolane": "O=S1(=O)CCCC1", "2-Glyme": "COCCOCCOC",
    "3-Glyme": "COCCOCCOCCOC", "4-Glyme": "COCCOCCOCCOCCOC", "3-Me-2-Oxazolidinone": "CN1CCOC1=O",
    "3-MeSulfolane": "CC1CCS(=O)(=O)C1", "Ethyldiglyme": "CCOCCOCCOCC", "DMF": "CN(C)C=O", "Ethylbenzene": "CCc1ccccc1",
    "Ethylmonoglyme": "CCOCCOCC", "Benzene": "c1ccccc1", "g-Butyrolactone": "O=C1CCCO1", "Cumene": "CC(C)c1ccccc1",
    "Propylsulfone": "CCCS(=O)(=O)CCC", "Pseudocumeme": "Cc1ccc(C)c(C)c1", "TEOS": "CCO[Si](OCC)(OCC)OCC",
    "m-Xylene": "Cc1cccc(C)c1", "o-Xylene": "Cc1ccccc1C",
}
ANION_SMILES = {  ## 키 = 대회 데이터의 salt 이름, 값 = 음이온
    "LiPF6": "F[P-](F)(F)(F)(F)F", "LiBF4": "F[B-](F)(F)F", "LiAsF6": "F[As-](F)(F)(F)(F)F", "LiClO4": "[O-]Cl(=O)(=O)=O",
    "LiCF3SO3": "[O-]S(=O)(=O)C(F)(F)F", "LiTFSI": "FC(F)(F)S(=O)(=O)[N-]S(=O)(=O)C(F)(F)F",
    "LiN(CF3SO2)2": "FC(F)(F)S(=O)(=O)[N-]S(=O)(=O)C(F)(F)F", "LiFSI": "O=S(=O)(F)[N-]S(=O)(=O)F",
    "LiBOB": "[B-]12(OC(=O)C(=O)O1)OC(=O)C(=O)O2", "LiBMB": "[B-]12(OC(=O)CC(=O)O1)OC(=O)CC(=O)O2",
    "LiBPFPB": "[B-]12(OC(C(F)(F)F)(C(F)(F)F)C(C(F)(F)F)(C(F)(F)F)O1)OC(C(F)(F)F)(C(F)(F)F)C(C(F)(F)F)(C(F)(F)F)O2",
    "LiTDI": "N#Cc1nc(C(F)(F)F)[n-]c1C#N", "LiPDI": "N#Cc1nc(C(F)(F)C(F)(F)F)[n-]c1C#N",
}
assert set(SOLV_SMILES) == set(SOLVENTS) and set(ANION_SMILES) == set(train["salt"].unique())

def rd_desc(smi):
    m = Chem.MolFromSmiles(smi); mh = Chem.AddHs(m)
    AllChem.ComputeGasteigerCharges(mh)
    q = np.array([a.GetDoubleProp("_GasteigerCharge") for a in mh.GetAtoms()], float)
    return {"MW": Descriptors.MolWt(m), "MR": Crippen.MolMR(m), "logP": Crippen.MolLogP(m),
            "TPSA": rdMolDescriptors.CalcTPSA(m), "HBA": rdMolDescriptors.CalcNumHBA(m),
            "nRot": rdMolDescriptors.CalcNumRotatableBonds(m), "fSP3": rdMolDescriptors.CalcFractionCSP3(m),
            "nHeavy": m.GetNumHeavyAtoms(), "nF": sum(a.GetSymbol() == "F" for a in m.GetAtoms()),
            "q_min": np.nanmin(q), "q_absmax": np.nanmax(np.abs(q))}   ## q_min: 가장 음전하가 큰 원자 (donor/염기성 근사)

def thermo_meta(cas):
    try:
        c = Chemical(cas)
    except Exception:
        return {"eta_method": None, "eps_method": None, "dipole": np.nan}
    vl = getattr(c, "ViscosityLiquid", None); pm = getattr(c, "Permittivity", None)
    return {"eta_method": getattr(vl, "method", None), "eps_method": getattr(pm, "method", None),
            "dipole": c.dipole if c.dipole is not None else np.nan}

SOLV_TAB = pd.DataFrame({s: rd_desc(smi) | thermo_meta(CAS[s]) |
                         {"B_lit": B_LIT.get(s, np.nan), "eps_a": EPS_A.get(s, np.nan), "eps_b": EPS_B.get(s, np.nan)}
                         for s, smi in SOLV_SMILES.items()}).T
ANION_TAB = pd.DataFrame({s: rd_desc(smi) for s, smi in ANION_SMILES.items()}).T
ANION_TAB["q_per_heavy"] = -1.0 / ANION_TAB["nHeavy"].astype(float)   ## 음전하 1을 무거운 원자 수로 나눔 (전하 비편재화 근사)

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print("== 용매 ==");  print(SOLV_TAB.to_string())
print("\n== 음이온 =="); print(ANION_TAB.round(3).to_string())
SOLV_TAB.to_csv("solvent_table.csv"); ANION_TAB.to_csv("anion_table.csv")

== 용매 ==
                           MW       MR     logP   TPSA  HBA nRot      fSP3 nHeavy   nF     q_min  q_absmax             eta_method    eps_method    dipole B_lit   eps_a    eps_b
EC                     88.062     17.3   0.1532  35.53    3    0  0.666667      6    0 -0.430657  0.508009           LETSOU_STIEL           CRC  4.510079   390 -57.082  155.184
PC                    102.089   21.895   0.5417  35.53    3    0      0.75      7    0 -0.430346   0.50829           LETSOU_STIEL           CRC       NaN   291  -8.647   72.174
DMC                    90.078   19.414   0.3992  35.53    3    0  0.666667      6    0 -0.437621  0.507198            REFPROP_FIT  CRC_CONSTANT       NaN  1079   3.087     -0.0
EMC                   104.105   24.031   0.7893  35.53    3    1      0.75      7    0 -0.437614  0.507467           LETSOU_STIEL  CRC_CONSTANT       NaN   475   2.985      0.0
DEC                   118.132   28.648   1.1794  35.53    3    2       0.8      8    0 -0.434588  0.507736

In [8]:
## [E-3] pandas feature: 기준 P0 (대회 노트북 C3T 구성을 몰분율 1개 기준으로 재현) + 후보 블록
import time
from sklearn.preprocessing import PolynomialFeatures

SOLV = {  ## C-3 값 그대로: (ε, η cP, MW, ρ, DN)
    "EC": (89.8, 1.90, 88.06, 1.32, 16.4), "PC": (64.9, 2.53, 102.09, 1.20, 15.1),
    "DMC": (3.1, 0.59, 90.08, 1.07, 17.2), "EMC": (2.96, 0.65, 104.10, 1.01, 17.0),
    "DEC": (2.8, 0.75, 118.13, 0.97, 16.0), "DME": (7.2, 0.46, 90.12, 0.87, 20.0),
    "DMSO": (46.5, 1.99, 78.13, 1.10, 29.8), "AN": (35.9, 0.34, 41.05, 0.78, 14.1),
    "MOEMC": (6.0, 1.10, 134.13, 1.08, 16.0), "TFP": (7.0, 2.30, 344.07, 1.59, 5.0),
    "EA": (6.0, 0.43, 88.11, 0.90, 17.1), "MA": (6.7, 0.36, 74.08, 0.93, 16.5),
    "FEC": (78.4, 4.10, 106.05, 1.45, 9.0), "DOL": (7.1, 0.59, 74.08, 1.06, 18.0),
    "2-MeTHF": (6.97, 0.46, 86.13, 0.85, 18.0), "DMM": (2.7, 0.33, 76.09, 0.86, 15.0),
    "Freon 11": (2.3, 0.42, 137.37, 1.49, 0.1), "Methylene chloride": (8.9, 0.41, 84.93, 1.33, 1.0),
    "THF": (7.6, 0.46, 72.11, 0.89, 20.0), "Toluene": (2.38, 0.56, 92.14, 0.87, 0.1),
    "Sulfolane": (43.3, 10.3, 120.17, 1.26, 14.8), "2-Glyme": (7.4, 0.99, 134.17, 0.94, 19.5),
    "3-Glyme": (7.5, 1.89, 178.23, 0.99, 14.0), "4-Glyme": (7.8, 3.39, 222.28, 1.01, 16.6),
    "3-Me-2-Oxazolidinone": (77.5, 2.45, 101.10, 1.17, 16.0), "3-MeSulfolane": (29.4, 11.0, 134.20, 1.20, 14.0),
    "Ethyldiglyme": (5.7, 1.40, 162.23, 0.91, 18.0), "DMF": (36.7, 0.80, 73.09, 0.94, 26.6),
    "Ethylbenzene": (2.4, 0.64, 106.17, 0.87, 0.1), "Ethylmonoglyme": (5.1, 0.65, 118.17, 0.84, 19.0),
    "Benzene": (2.28, 0.60, 78.11, 0.88, 0.1), "g-Butyrolactone": (39.0, 1.73, 86.09, 1.13, 18.0),
    "Cumene": (2.38, 0.74, 120.19, 0.86, 0.1), "Propylsulfone": (32.0, 4.00, 150.24, 1.03, 14.0),
    "Pseudocumeme": (2.38, 0.89, 120.19, 0.88, 0.1), "TEOS": (2.5, 0.70, 208.33, 0.93, 5.0),
    "m-Xylene": (2.37, 0.58, 106.17, 0.86, 0.1), "o-Xylene": (2.57, 0.76, 106.17, 0.88, 0.1),
}
SALT = {  ## C-3 값 그대로: (MW, 음이온 부피, 해리도 점수)
    "LiPF6": (151.9, 69, 0.8), "LiBOB": (193.8, 120, 0.6), "LiBF4": (93.7, 53, 0.4),
    "LiAsF6": (195.9, 72, 0.85), "LiBPFPB": (671.0, 330, 0.7), "LiCF3SO3": (156.0, 80, 0.2),
    "LiClO4": (106.4, 55, 0.6), "LiTFSI": (287.1, 150, 0.9), "LiN(CF3SO2)2": (287.1, 150, 0.9),
    "LiBMB": (208.0, 140, 0.5), "LiFSI": (187.1, 100, 0.9), "LiTDI": (192.0, 110, 0.75),
    "LiPDI": (242.0, 130, 0.75),
}
ANDRADE_B = {  ## C-14 값 그대로 (K)
    "EC": 1800, "PC": 1900, "DMC": 1000, "EMC": 1100, "DEC": 1200, "DME": 950, "DMSO": 1350, "AN": 850,
    "MOEMC": 1300, "TFP": 1800, "EA": 950, "MA": 850, "FEC": 1900, "DOL": 1000, "2-MeTHF": 950, "DMM": 800,
    "Freon 11": 800, "Methylene chloride": 820, "THF": 850, "Toluene": 1050, "Sulfolane": 2200,
    "2-Glyme": 1200, "3-Glyme": 1550, "4-Glyme": 1900, "3-Me-2-Oxazolidinone": 1700, "3-MeSulfolane": 2300,
    "Ethyldiglyme": 1350, "DMF": 900, "Ethylbenzene": 1100, "Ethylmonoglyme": 1000, "Benzene": 1200,
    "g-Butyrolactone": 1400, "Cumene": 1200, "Propylsulfone": 1900, "Pseudocumeme": 1250, "TEOS": 1250,
    "m-Xylene": 1050, "o-Xylene": 1150,
}
N2 = {  ## C-14 값 그대로 (ε∞ ≈ n²)
    "EC": 2.02, "PC": 2.02, "DMC": 1.87, "EMC": 1.89, "DEC": 1.92, "DME": 1.91, "DMSO": 2.19, "AN": 1.81,
    "MOEMC": 1.95, "TFP": 1.74, "EA": 1.88, "MA": 1.85, "FEC": 1.95, "DOL": 1.95, "2-MeTHF": 1.98, "DMM": 1.83,
    "Freon 11": 1.92, "Methylene chloride": 2.03, "THF": 1.97, "Toluene": 2.24, "Sulfolane": 2.21,
    "2-Glyme": 1.97, "3-Glyme": 2.02, "4-Glyme": 2.05, "3-Me-2-Oxazolidinone": 2.10, "3-MeSulfolane": 2.20,
    "Ethyldiglyme": 1.98, "DMF": 2.04, "Ethylbenzene": 2.24, "Ethylmonoglyme": 1.93, "Benzene": 2.25,
    "g-Butyrolactone": 2.06, "Cumene": 2.23, "Propylsulfone": 2.10, "Pseudocumeme": 2.27, "TEOS": 1.91,
    "m-Xylene": 2.24, "o-Xylene": 2.26,
}

col = lambda d, j: np.array([d[s][j] for s in SOLVENTS], float)
EPS25, ETA25, MWS, RHO, DN = (col(SOLV, j) for j in range(5))
BAND = np.array([ANDRADE_B[s] for s in SOLVENTS], float); NN = np.array([N2[s] for s in SOLVENTS], float)

## 분율 → 몰분율 (행의 분율 기준에 맞춰 환산)
Fr = X[SOLVENTS].to_numpy(float)
rt = X["solvent ratio type"].astype(str).to_numpy()
g = np.where((rt == "w")[:, None], 1 / MWS, np.where((rt == "v")[:, None], RHO / MWS, 1.0))
nmol = Fr * g; XM = nmol / nmol.sum(1, keepdims=True)

T = X["T"].to_numpy(float); C = X["c"].to_numpy(float)
salt = X["salt"].astype(str).to_numpy()
lnetaT = XM @ np.log(ETA25) + (XM @ BAND) * (1 / T - 1 / 298.15)        ## Andrade, 몰분율 혼합
epsT = XM @ NN + (XM @ (EPS25 - NN)) * 298.15 / T                      ## ε(T) = n² + (ε25 − n²)·298.15/T
SD = pd.DataFrame(SALT, index=["s_MW", "s_vol", "s_dissoc"]).T.loc[salt].reset_index(drop=True)

## ── 기준 P0
core = pd.DataFrame({"invT": 1000 / T, "invT150": 1000 / (T - 150), "lnc": np.log(C + 0.01), "sqc": np.sqrt(C), "c": C,
                     "lnetaT": lnetaT, "lnepsT": np.log(epsT), "dissoc": SD["s_dissoc"], "vol": SD["s_vol"] / 100})
pf = PolynomialFeatures(2, include_bias=False)
CORE2 = pd.DataFrame(pf.fit_transform(core), columns=pf.get_feature_names_out(core.columns))
P0 = pd.concat([pd.get_dummies(X[CATEGORICAL], dtype=float).reset_index(drop=True),
                X[["T", "c"] + SOLVENTS].reset_index(drop=True),
                pd.DataFrame({"m_lneta25": XM @ np.log(ETA25), "m_eps25": XM @ EPS25, "m_DN": XM @ DN,
                              "m_MW": XM @ MWS, "m_rho": XM @ RHO}),
                SD, CORE2], axis=1)

## ── 후보 블록
RDS = SOLV_TAB.loc[SOLVENTS, ["MR", "logP", "TPSA", "HBA", "nRot", "fSP3", "q_min"]].astype(float)
B_rd_solv = pd.DataFrame(XM @ RDS.to_numpy(), columns=["rs_" + c for c in RDS.columns])

AT = ANION_TAB[["MR", "nHeavy", "nF", "TPSA", "logP", "q_min", "q_per_heavy"]].astype(float).copy()
AT["q_min"] = AT["q_min"].fillna(AT["q_min"].max())   ## PF6⁻·AsF6⁻: Gasteiger 실패 → 가장 약한 염기성 값 (약배위 음이온 가정)
B_rd_anion = AT.loc[salt].reset_index(drop=True).add_prefix("an_")

def wstd(v):  ## 몰분율 가중 표준편차 (혼합물 안의 퍼짐)
    m = XM @ v
    return np.sqrt(np.clip(XM @ (v ** 2) - m ** 2, 0, None))
present = Fr > 0
B_hetero = pd.DataFrame({"h_std_eps": wstd(EPS25), "h_std_lneta": wstd(np.log(ETA25)), "h_std_DN": wstd(DN),
                         "h_max_DN": np.where(present, DN, -np.inf).max(1),
                         "h_max_eps": np.where(present, EPS25, -np.inf).max(1),
                         "h_frac_hieps": XM @ (EPS25 > 20).astype(float)})

qmin = B_rd_anion["an_q_min"].to_numpy(); qph = B_rd_anion["an_q_per_heavy"].to_numpy(); amr = B_rd_anion["an_MR"].to_numpy()
lnc, lneT = core["lnc"].to_numpy(), core["lnepsT"].to_numpy()
B_pair = pd.DataFrame({"p_qmin_DN": qmin * (XM @ DN), "p_qmin_lneps": qmin * lneT, "p_qph_lneps": qph * lneT,
                       "p_qmin_lnc": qmin * lnc, "p_MR_lnc": amr * lnc, "p_MR_lneta": amr * lnetaT})

VARIANTS = {
    "P0": P0,
    "P0+rd_solv": pd.concat([P0, B_rd_solv], axis=1),
    "P0+rd_anion": pd.concat([P0, B_rd_anion], axis=1),
    "P0+hetero": pd.concat([P0, B_hetero], axis=1),
    "P0+pair": pd.concat([P0, B_rd_anion, B_pair], axis=1),
    "P0+all": pd.concat([P0, B_rd_solv, B_rd_anion, B_hetero, B_pair], axis=1),
    "P0-saltOH": P0.drop(columns=[c for c in P0.columns if c.startswith("salt_")]),
}
for n, F in VARIANTS.items():
    print(f"{n:14s} {F.shape} | finite {np.isfinite(F.to_numpy(float)).all()}")

P0             (10810, 120) | finite True
P0+rd_solv     (10810, 127) | finite True
P0+rd_anion    (10810, 127) | finite True
P0+hetero      (10810, 126) | finite True
P0+pair        (10810, 133) | finite True
P0+all         (10810, 146) | finite True
P0-saltOH      (10810, 107) | finite True


In [9]:
## [E-4] 후보 블록 비교: 같은 분할·같은 모델(대회 노트북 LGBL 설정) → 구간별 점수 + mol/l unseen fold별 Δ
from lightgbm import LGBMRegressor
LGBP = dict(n_estimators=1500, learning_rate=0.02, num_leaves=15, min_child_samples=40, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.6, reg_lambda=5.0, linear_tree=True, linear_lambda=1.0, n_jobs=-1, random_state=0, verbose=-1)
RES = globals().get("RES", {})   ## 재실행 시 이미 계산한 변형은 건너뜀

def oof(F):
    A = F.to_numpy(float); o = np.zeros(len(y))
    for fi, vi in SPLITS:
        m = LGBMRegressor(**LGBP).fit(A[fi], LY[fi])
        o[vi] = np.clip(m.predict(A[vi]), 0, None)
    return o

for n, F in VARIANTS.items():
    if n not in RES:
        t0 = time.time(); RES[n] = oof(F); print(f"{n:14s} {F.shape[1]:4d}열 {time.time() - t0:4.0f}s")

seg = lambda o, m: np.sqrt(((o - LY)[m] ** 2).mean())
U = SEG["mol/l unseen"]
fu = lambda o: np.array([np.sqrt(((o[vi] - LY[vi])[U[vi]] ** 2).mean()) for _, vi in SPLITS])
rows = {}
for n, o in RES.items():
    r = {"cv": np.mean([np.sqrt(((o[vi] - LY[vi]) ** 2).mean()) for _, vi in SPLITS])} | {k: seg(o, m) for k, m in SEG.items()}
    d = fu(o) - fu(RES["P0"])
    r |= {"Δunseen": r["mol/l unseen"] - seg(RES["P0"], U), "개선 fold": f"{int((d < 0).sum())}/5", "fold Δ": np.round(d, 3)}
    rows[n] = r
pd.set_option("display.width", 250)
print(pd.DataFrame(rows).T.round(4).to_string())

P0              120열   48s
P0+rd_solv      127열   38s
P0+rd_anion     127열   39s
P0+hetero       126열   37s
P0+pair         133열   40s
P0+all          146열   43s
P0-saltOH       107열   36s
                   cv    mol/kg mol/l seen mol/l unseen   Δunseen 개선 fold                                  fold Δ
P0           0.176304  0.150403   0.296122     0.226255       0.0     0/5               [0.0, 0.0, 0.0, 0.0, 0.0]
P0+rd_solv   0.168465  0.151492   0.237369     0.222837 -0.003418     3/5   [0.029, -0.007, 0.02, -0.039, -0.017]
P0+rd_anion  0.169472  0.152241   0.229237     0.231045   0.00479     3/5    [-0.006, -0.0, 0.004, 0.021, -0.004]
P0+hetero     0.16786  0.150716   0.243246     0.215316 -0.010939     3/5  [0.016, -0.005, 0.001, -0.042, -0.014]
P0+pair      0.169494  0.152075   0.228794     0.232463  0.006208     2/5   [-0.012, -0.003, 0.013, 0.014, 0.003]
P0+all        0.16513  0.151578   0.213589     0.216658 -0.009597     3/5     [0.007, 0.0, -0.002, -0.02, -0.029]
P0-saltOH    

In [10]:
## [E-5] 진단: (1) P0의 mol/l seen 오차가 소수 행에 몰렸는가  (2) seed만 바꿨을 때의 흔들림 = 노이즈 바닥
S = SEG["mol/l seen"]
se0 = (RES["P0"] - LY) ** 2
top = np.argsort(-np.where(S, se0, -1.0))[:10]
print(pd.DataFrame({"salt": salt[top], "T": T[top], "c": C[top], "k": y.to_numpy()[top],
                    "pred": np.expm1(RES["P0"][top]), "SE": se0[top]}).round(3).to_string())
print("mol/l seen SE 중 상위 10행 비중:", round(se0[top].sum() / se0[S].sum(), 3))

def oof_seed(F, seed):
    A = F.to_numpy(float); o = np.zeros(len(y))
    for fi, vi in SPLITS:
        m = LGBMRegressor(**(LGBP | {"random_state": seed})).fit(A[fi], LY[fi])
        o[vi] = np.clip(m.predict(A[vi]), 0, None)
    return o

for seed in [1, 2]:
    for n in ["P0", "P0+hetero"]:
        key = f"{n}@s{seed}"
        if key not in RES:
            t0 = time.time(); RES[key] = oof_seed(VARIANTS[n], seed); print(f"{key:16s} {time.time() - t0:4.0f}s")

keys = ["P0", "P0@s1", "P0@s2", "P0+hetero", "P0+hetero@s1", "P0+hetero@s2"]
print(pd.DataFrame({k: {g: seg(RES[k], m) for g, m in SEG.items()} for k in keys}).T.round(4).to_string())

      salt        T    c      k    pred     SE
0   LiAsF6  333.150  2.0  2.060  13.793  2.483
1  LiBPFPB  232.320  0.6  1.196   8.965  2.288
2  LiBPFPB  237.008  0.6  1.498  10.118  2.229
3  LiBPFPB  242.175  0.6  1.858  11.673  2.218
4  LiBPFPB  247.422  0.6  2.351  13.073  2.059
5  LiBPFPB  222.754  0.6  0.717   6.006  1.978
6  LiBPFPB  227.309  0.6  0.955   6.969  1.975
7  LiBPFPB  253.058  0.6  2.960  15.138  1.974
8  LiBPFPB  217.221  0.6  0.544   5.181  1.925
9  LiBPFPB  257.656  0.6  3.708  17.286  1.841
mol/l seen SE 중 상위 10행 비중: 0.221
P0@s1              40s
P0+hetero@s1       40s
P0@s2              41s
P0+hetero@s2       37s
              mol/kg  mol/l seen  mol/l unseen
P0            0.1504      0.2961        0.2263
P0@s1         0.1505      0.2095        0.2321
P0@s2         0.1511      0.2217        0.2303
P0+hetero     0.1507      0.2432        0.2153
P0+hetero@s1  0.1501      0.2804        0.2191
P0+hetero@s2  0.1504      0.2402        0.2150


In [11]:
## [E-6] P0의 mol/kg 우위는 어디서 오는가: P0를 대회 노트북 C3T 구성 쪽으로 한 요소씩 바꾸는 ablation
GB = [np.ones_like(MWS), 1 / MWS, RHO / MWS]                        ## 대회 노트북 W1의 3개 기준 (원본 분율 / w→mol / v→mol)
XB = [(Fr * gb) / (Fr * gb).sum(1, keepdims=True) for gb in GB]

def core_3b():  ## 물리 핵심을 3개 기준으로 (대회 노트북 phys3_block("T")과 같은 13열 → 2차 확장)
    d = {"invT": 1000 / T, "invT150": 1000 / (T - 150), "lnc": np.log(C + 0.01), "sqc": np.sqrt(C), "c": C}
    for b, xb in enumerate(XB):
        d[f"lnetaT_{b}"] = xb @ np.log(ETA25) + (xb @ BAND) * (1 / T - 1 / 298.15)
    for b, xb in enumerate(XB):
        d[f"lnepsT_{b}"] = np.log(xb @ NN + (xb @ (EPS25 - NN)) * 298.15 / T)
    d["dissoc"] = SD["s_dissoc"].to_numpy(); d["vol"] = SD["s_vol"].to_numpy() / 100
    core = pd.DataFrame(d); pfb = PolynomialFeatures(2, include_bias=False)
    return pd.DataFrame(pfb.fit_transform(core), columns=pfb.get_feature_names_out(core.columns))

PROPS7 = np.c_[EPS25, np.log(ETA25) + 3, np.log(EPS25), DN + 1, MWS, RHO, 100 / MWS]   ## 대회 노트북 C-3의 props와 같음
def mix7_3b():  ## 기본 혼합 descriptor를 3개 기준 × 7개 log 평균으로 (대회 노트북 make_features의 "solv"와 같음)
    return pd.DataFrame(np.hstack([np.log(xb @ PROPS7) for xb in XB]),
                        columns=[f"mix7_b{b}_{j}" for b in range(3) for j in range(7)])

ONEHOT = pd.get_dummies(X[CATEGORICAL], dtype=float).reset_index(drop=True)
RAW = X[["T", "c"] + SOLVENTS].reset_index(drop=True)
MIX5 = pd.DataFrame({"m_lneta25": XM @ np.log(ETA25), "m_eps25": XM @ EPS25, "m_DN": XM @ DN,
                     "m_MW": XM @ MWS, "m_rho": XM @ RHO})
C3B, M7B = core_3b(), mix7_3b()
ABL = {
    "A_core3b": pd.concat([ONEHOT, RAW, MIX5, SD, C3B], axis=1),
    "A_mix3b": pd.concat([ONEHOT, RAW, M7B, SD, CORE2], axis=1),
    "A_both3b": pd.concat([ONEHOT, RAW, M7B, SD, C3B], axis=1),     ## C3T 재현 대조군
}
for n, F in ABL.items():
    if n not in RES:
        t0 = time.time(); RES[n] = oof(F); print(f"{n:10s} {F.shape[1]:4d}열 {time.time() - t0:4.0f}s")

NCOL = {"P0": P0.shape[1]} | {n: F.shape[1] for n, F in ABL.items()}
tab = pd.DataFrame({n: {"열 수": NCOL[n], "cv": np.mean([np.sqrt(((RES[n][vi] - LY[vi]) ** 2).mean()) for _, vi in SPLITS])}
                       | {g: seg(RES[n], m) for g, m in SEG.items()} for n in NCOL}).T
print(tab.round(4).to_string())

A_core3b    170열   75s
A_mix3b     136열   40s
A_both3b    186열   62s
            열 수      cv  mol/kg  mol/l seen  mol/l unseen
P0        120.0  0.1763  0.1504      0.2961        0.2263
A_core3b  170.0  0.1717  0.1515      0.2521        0.2309
A_mix3b   136.0  0.1666  0.1498      0.2319        0.2206
A_both3b  186.0  0.1676  0.1516      0.2185        0.2293


In [12]:
## [E-7] 계 균형 가중치: w = 1 / n_계^α (α = 0, 0.5, 1) × seed 2개 — 기준 feature는 C3T 재현(A_both3b)
def oof_w(F, alpha, seed):
    A = F.to_numpy(float); o = np.zeros(len(y))
    for fi, vi in SPLITS:
        s_fi = pd.Series(SYS[fi])
        w = s_fi.map(s_fi.value_counts()).to_numpy(float) ** (-alpha)    ## 학습 fold 안에서 센 계의 행 수
        w = w / w.mean()                                                 ## 평균 1로 정규화
        m = LGBMRegressor(**(LGBP | {"random_state": seed})).fit(A[fi], LY[fi], sample_weight=w)
        o[vi] = np.clip(m.predict(A[vi]), 0, None)
    return o

FB = ABL["A_both3b"]
ALPHAS, SEEDS = [0.0, 0.5, 1.0], [0, 1]
for alpha in ALPHAS:
    for sd in SEEDS:
        key = f"W{alpha}@s{sd}"
        if key not in RES:
            t0 = time.time(); RES[key] = oof_w(FB, alpha, sd); print(f"{key:10s} {time.time() - t0:4.0f}s")

print("확인 (α=0, seed 0 == A_both3b):", np.allclose(RES["W0.0@s0"], RES["A_both3b"]))

## 1) 구간별 seed 평균 / seed 간 범위 / α=0 대비 Δ
rows = {}
for alpha in ALPHAS:
    r = {}
    for g, m in SEG.items():
        v = [seg(RES[f"W{alpha}@s{sd}"], m) for sd in SEEDS]
        r[g] = np.mean(v); r[g + " 범위"] = max(v) - min(v)
    rows[f"α={alpha}"] = r
tab = pd.DataFrame(rows).T
for g in SEG: tab[g + " Δ"] = tab[g] - tab.loc["α=0.0", g]
print(tab[[c for g in SEG for c in (g, g + " Δ", g + " 범위")]].round(4).to_string())

## 2) fold별 Δ (seed 평균, α=0 대비) — mol/l 두 구간
for g in ["mol/l seen", "mol/l unseen"]:
    m = SEG[g]
    ff = lambda k: np.array([np.sqrt(((RES[k][vi] - LY[vi])[m[vi]] ** 2).mean()) for _, vi in SPLITS])
    base = np.mean([ff(f"W0.0@s{sd}") for sd in SEEDS], axis=0)
    for alpha in ALPHAS[1:]:
        d = np.mean([ff(f"W{alpha}@s{sd}") for sd in SEEDS], axis=0) - base
        print(f"{g:12s} α={alpha}: fold Δ {np.round(d, 3)} | 개선 fold {(d < 0).sum()}/5")

W0.0@s0      71s
W0.0@s1      63s
W0.5@s0      60s
W0.5@s1      58s
W1.0@s0      62s
W1.0@s1      62s
확인 (α=0, seed 0 == A_both3b): True
       mol/kg  mol/kg Δ  mol/kg 범위  mol/l seen  mol/l seen Δ  mol/l seen 범위  mol/l unseen  mol/l unseen Δ  mol/l unseen 범위
α=0.0  0.1516    0.0000     0.0000      0.2205        0.0000         0.0041        0.2290          0.0000           0.0006
α=0.5  0.1488   -0.0028     0.0001      0.2058       -0.0147         0.0026        0.2200         -0.0090           0.0012
α=1.0  0.1490   -0.0026     0.0001      0.1904       -0.0301         0.0143        0.2253         -0.0037           0.0034
mol/l seen   α=0.5: fold Δ [-0.054  0.02   0.01  -0.052  0.002] | 개선 fold 2/5
mol/l seen   α=1.0: fold Δ [-0.106  0.004  0.008 -0.066  0.001] | 개선 fold 2/5
mol/l unseen α=0.5: fold Δ [-0.003 -0.003  0.01  -0.04  -0.014] | 개선 fold 4/5
mol/l unseen α=1.0: fold Δ [ 0.018  0.006  0.02  -0.05  -0.01 ] | 개선 fold 2/5


In [13]:
## [E-8] outlier 부드러운 가중치: w = (계 균형 α=0.5) × (이탈 행이면 β) — β = 1(E-7), 0.5, 0.2 × seed 2개
def outlier_flag(idx):  ## idx 행들 안에서: 같은 계·같은 T의 c 방향 이웃 7개 중앙값과 log 차이 > 0.5
    d = pd.DataFrame({"sys": SYS[idx], "T": np.round(T[idx], 1), "c": C[idx], "ly": LY[idx], "i": np.arange(len(idx))})
    d = d.sort_values(["sys", "T", "c"])
    med = d.groupby(["sys", "T"])["ly"].transform(lambda s: s.rolling(7, center=True, min_periods=3).median())
    flag = ((d["ly"] - med).abs() > 0.5).to_numpy()
    out = np.zeros(len(idx), bool); out[d["i"].to_numpy()] = flag
    return out

FLAG_ALL = outlier_flag(np.arange(len(y)))
print("전체 기준 이탈 행:", int(FLAG_ALL.sum()), "| 그중 mol/kg:", int((FLAG_ALL & IS_KG).sum()), "← 이전 정리의 약 199행과 비슷해야 함")

def oof_wo(F, alpha, beta, seed):
    A = F.to_numpy(float); o = np.zeros(len(y))
    for fi, vi in SPLITS:
        s_fi = pd.Series(SYS[fi])
        w = s_fi.map(s_fi.value_counts()).to_numpy(float) ** (-alpha)
        w = w * np.where(outlier_flag(fi), beta, 1.0)           ## 판정은 학습 fold 라벨만 사용
        w = w / w.mean()
        m = LGBMRegressor(**(LGBP | {"random_state": seed})).fit(A[fi], LY[fi], sample_weight=w)
        o[vi] = np.clip(m.predict(A[vi]), 0, None)
    return o

BETAS = [0.5, 0.2]
for beta in BETAS:
    for sd in SEEDS:
        key = f"W0.5_b{beta}@s{sd}"
        if key not in RES:
            t0 = time.time(); RES[key] = oof_wo(FB, 0.5, beta, sd); print(f"{key:16s} {time.time() - t0:4.0f}s")

## 구간별 seed 평균 / 범위 / β=1 대비 Δ (mol/kg은 정상·이탈로 나눔)
SEG2 = {"전체 cv": None, "mol/kg 정상": IS_KG & ~FLAG_ALL, "mol/kg 이탈": IS_KG & FLAG_ALL,
        "mol/l seen": SEG["mol/l seen"], "mol/l unseen": SEG["mol/l unseen"]}
cvf = lambda o: np.mean([np.sqrt(((o[vi] - LY[vi]) ** 2).mean()) for _, vi in SPLITS])
GROUPS = {"β=1": [f"W0.5@s{sd}" for sd in SEEDS]} | {f"β={b}": [f"W0.5_b{b}@s{sd}" for sd in SEEDS] for b in BETAS}
rows = {}
for lab, ks in GROUPS.items():
    r = {}
    for g, m in SEG2.items():
        v = [cvf(RES[k]) if m is None else seg(RES[k], m) for k in ks]
        r[g] = np.mean(v); r[g + " 범위"] = max(v) - min(v)
    rows[lab] = r
tab = pd.DataFrame(rows).T
for g in SEG2: tab[g + " Δ"] = tab[g] - tab.loc["β=1", g]
print(tab[[c for g in SEG2 for c in (g, g + " Δ", g + " 범위")]].round(4).to_string())

## 전체 cv의 fold별 Δ (seed 평균, β=1 대비)
ffold = lambda k: np.array([np.sqrt(((RES[k][vi] - LY[vi]) ** 2).mean()) for _, vi in SPLITS])
base = np.mean([ffold(k) for k in GROUPS["β=1"]], axis=0)
for b in BETAS:
    d = np.mean([ffold(k) for k in GROUPS[f"β={b}"]], axis=0) - base
    print(f"β={b}: 전체 fold Δ {np.round(d, 4)} | 개선 fold {(d < 0).sum()}/5")

전체 기준 이탈 행: 259 | 그중 mol/kg: 199 ← 이전 정리의 약 199행과 비슷해야 함
W0.5_b0.5@s0       73s
W0.5_b0.5@s1       74s
W0.5_b0.2@s0       73s
W0.5_b0.2@s1       67s
        전체 cv  전체 cv Δ  전체 cv 범위  mol/kg 정상  mol/kg 정상 Δ  mol/kg 정상 범위  mol/kg 이탈  mol/kg 이탈 Δ  mol/kg 이탈 범위  mol/l seen  mol/l seen Δ  mol/l seen 범위  mol/l unseen  mol/l unseen Δ  mol/l unseen 범위
β=1    0.1627   0.0000    0.0003     0.1043       0.0000        0.0000     0.7009       0.0000        0.0003      0.2058        0.0000         0.0026        0.2200          0.0000           0.0012
β=0.5  0.1634   0.0007    0.0024     0.0949      -0.0094        0.0016     0.7502       0.0493        0.0038      0.2133        0.0075         0.0286        0.2208          0.0009           0.0021
β=0.2  0.1676   0.0049    0.0023     0.0893      -0.0150        0.0011     0.7986       0.0976        0.0021      0.2401        0.0343         0.0264        0.2172         -0.0028           0.0007
β=0.5: 전체 fold Δ [ 0.0044  0.0017 -0.0016  0.     -0.0011] | 개선

In [14]:
## [E-9] 물리 baseline 위의 잔차 학습: ln(1+k) = Ridge(매끄러운 물리 기저) + LGBM(잔차)
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline as mkp

lnc_, invT_ = np.log(C + 0.01), 1000 / T
BASIS = pd.concat([
    pd.DataFrame({"lnc": lnc_, "c": C, "sqc": np.sqrt(C), "invT": invT_, "invT150": 1000 / (T - 150),
                  "lnetaT": lnetaT, "lnepsT": np.log(epsT),
                  "lnc_invT": lnc_ * invT_, "lnc_lneta": lnc_ * lnetaT, "c_lneta": C * lnetaT, "c_invT": C * invT_}),
    pd.get_dummies(X[["salt", "c units"]], dtype=float).reset_index(drop=True)], axis=1)   ## 염·단위: 절편 이동만

def oof_resid(F, seed, linear_leaf, resid=True, alpha=0.5):
    A, Bm = F.to_numpy(float), BASIS.to_numpy(float)
    o, ob = np.zeros(len(y)), np.zeros(len(y))
    for fi, vi in SPLITS:
        s_fi = pd.Series(SYS[fi]); w = s_fi.map(s_fi.value_counts()).to_numpy(float) ** (-alpha); w = w / w.mean()
        if resid:
            base = mkp(StandardScaler(), Ridge(alpha=1.0)).fit(Bm[fi], LY[fi], ridge__sample_weight=w)
            bf, bv = base.predict(Bm[fi]), base.predict(Bm[vi])
        else:
            bf, bv = np.zeros(len(fi)), np.zeros(len(vi))
        m = LGBMRegressor(**(LGBP | {"random_state": seed, "linear_tree": linear_leaf})).fit(A[fi], LY[fi] - bf, sample_weight=w)
        ob[vi] = np.clip(bv, 0, None)
        o[vi] = np.clip(bv + m.predict(A[vi]), 0, None)
    return o, ob

VAR9 = {"R_lin": dict(linear_leaf=True, resid=True), "G_ctrl": dict(linear_leaf=False, resid=False),
        "R_gbdt": dict(linear_leaf=False, resid=True)}
for n, kw in VAR9.items():
    for sd in SEEDS:
        key = f"{n}@s{sd}"
        if key not in RES:
            t0 = time.time(); RES[key], ob = oof_resid(FB, sd, **kw); print(f"{key:12s} {time.time() - t0:4.0f}s")
            if kw["resid"] and "R_base" not in RES: RES["R_base"] = ob

## 구간별 seed 평균 / 범위 / 기존(E-7 α=0.5) 대비 Δ
SEG3 = {"전체 cv": None, "mol/kg": SEG["mol/kg"], "mol/l seen": SEG["mol/l seen"], "mol/l unseen": SEG["mol/l unseen"]}
G9 = {"기존 (선형 리프)": [f"W0.5@s{sd}" for sd in SEEDS]} | {n: [f"{n}@s{sd}" for sd in SEEDS] for n in VAR9} | {"baseline 단독": ["R_base"]}
rows = {}
for lab, ks in G9.items():
    r = {}
    for g, m in SEG3.items():
        v = [cvf(RES[k]) if m is None else seg(RES[k], m) for k in ks]
        r[g] = np.mean(v); r[g + " 범위"] = max(v) - min(v)
    rows[lab] = r
tab = pd.DataFrame(rows).T
for g in SEG3: tab[g + " Δ"] = tab[g] - tab.loc["기존 (선형 리프)", g]
print(tab[[c for g in SEG3 for c in (g, g + " Δ", g + " 범위")]].round(4).to_string())

base9 = np.mean([ffold(k) for k in G9["기존 (선형 리프)"]], axis=0)
for n in VAR9:
    d = np.mean([ffold(k) for k in G9[n]], axis=0) - base9
    print(f"{n:7s}: 전체 fold Δ {np.round(d, 4)} | 개선 fold {(d < 0).sum()}/5")

R_lin@s0       75s
R_lin@s1       62s
G_ctrl@s0      62s
G_ctrl@s1      54s
R_gbdt@s0      56s
R_gbdt@s1      53s
              전체 cv  전체 cv Δ  전체 cv 범위  mol/kg  mol/kg Δ  mol/kg 범위  mol/l seen  mol/l seen Δ  mol/l seen 범위  mol/l unseen  mol/l unseen Δ  mol/l unseen 범위
기존 (선형 리프)   0.1627   0.0000    0.0003  0.1488    0.0000     0.0001      0.2058        0.0000         0.0026        0.2200          0.0000           0.0012
R_lin        0.1645   0.0018    0.0008  0.1460   -0.0027     0.0006      0.2037       -0.0021         0.0024        0.2433          0.0234           0.0045
G_ctrl       0.1620  -0.0007    0.0017  0.1471   -0.0017     0.0002      0.2059        0.0001         0.0043        0.2217          0.0017           0.0064
R_gbdt       0.1666   0.0039    0.0007  0.1430   -0.0058     0.0006      0.2332        0.0273         0.0034        0.2448          0.0248           0.0007
baseline 단독  0.3087   0.1461    0.0000  0.2543    0.1055     0.0000      0.5776        0.3718         0.00

In [ ]:
## [E-10] 물리적으로 제한된 baseline(B1) + 단위별 baseline(B2) + baseline을 feature로도 제공(B3)
LNK = np.log(np.clip(np.expm1(LY), 1e-3, None))                      ## 물리식은 ln k 공간에서 맞춤
BASIS2 = pd.concat([pd.DataFrame({"lnc": lnc_, "c": C, "lnetaT": lnetaT, "lnepsT": np.log(epsT), "invT": invT_}),
                    pd.get_dummies(X[["salt"]], dtype=float).reset_index(drop=True)], axis=1)
## 단일 변수 단조 항만: ln c(멱법칙), c(고농도 지수 감쇠), ln η(T)(Walden), ln ε(T), 1/T(Arrhenius) + 염 절편
B2M = BASIS2.to_numpy(float)

def fit_baseline(fi, w, per_unit=True):  ## 학습 행 fi로 ln k baseline을 맞추고, 행 idx의 ln(1+k) 예측 함수를 반환
    parts = [IS_KG, ~IS_KG] if per_unit else [np.ones(len(y), bool)]
    models = []
    for gm in parts:
        sel = gm[fi]
        m = mkp(StandardScaler(), Ridge(alpha=1.0)).fit(B2M[fi][sel], LNK[fi][sel], ridge__sample_weight=w[sel])
        models.append((gm, m))
    def predict(idx):
        out = np.zeros(len(idx))
        for gm, m in models:
            s = gm[idx]
            if s.any(): out[s] = m.predict(B2M[idx][s])
        return np.log1p(np.exp(out))                                   ## ln k → ln(1+k)
    return predict

def fold_w(fi, alpha=0.5):
    s_fi = pd.Series(SYS[fi]); w = s_fi.map(s_fi.value_counts()).to_numpy(float) ** (-alpha)
    return w / w.mean()

def oof_base(per_unit):  ## baseline 단독 OOF (진단용)
    ob = np.zeros(len(y))
    for fi, vi in SPLITS:
        ob[vi] = fit_baseline(fi, fold_w(fi), per_unit)(vi)
    return ob

def oof_resid2(F, seed, linear_leaf, base_as_feat=False):
    A = F.to_numpy(float); o = np.zeros(len(y))
    for fi, vi in SPLITS:
        w = fold_w(fi); bpred = fit_baseline(fi, w, True)
        bf, bv = bpred(fi), bpred(vi)
        Af, Av = (np.c_[A[fi], bf], np.c_[A[vi], bv]) if base_as_feat else (A[fi], A[vi])
        m = LGBMRegressor(**(LGBP | {"random_state": seed, "linear_tree": linear_leaf})).fit(Af, LY[fi] - bf, sample_weight=w)
        o[vi] = np.clip(bv + m.predict(Av), 0, None)
    return o

## 1) baseline 단독 진단 (문헌 권고: 물리 모델 성능을 먼저 확인)
for n, pu in [("B1_glob", False), ("B1_unit", True)]:
    if n not in RES: RES[n] = oof_base(pu)
print(pd.DataFrame({n: {"전체 cv": cvf(RES[n])} | {g: seg(RES[n], m) for g, m in SEG.items()}
                    for n in ["R_base", "B1_glob", "B1_unit"]}).T.round(4).to_string())
print("↑ 확인 지점: B1_unit의 mol/l seen이 0.3 아래면 baseline이 mol/l에서 쓸 만한 수준\n")

## 2) 잔차 학습 3변형 × seed 2개
VAR10 = {"R2_gbdt": dict(linear_leaf=False), "R2_lin": dict(linear_leaf=True),
         "R2_gbdt_feat": dict(linear_leaf=False, base_as_feat=True)}
for n, kw in VAR10.items():
    for sd in SEEDS:
        key = f"{n}@s{sd}"
        if key not in RES:
            t0 = time.time(); RES[key] = oof_resid2(FB, sd, **kw); print(f"{key:16s} {time.time() - t0:4.0f}s")

## 3) 구간별 seed 평균 / 범위 / 기존(E-7 α=0.5, 선형 리프) 대비 Δ
G10 = {"기존 (선형 리프)": [f"W0.5@s{sd}" for sd in SEEDS], "R_gbdt (E-9)": [f"R_gbdt@s{sd}" for sd in SEEDS]} \
      | {n: [f"{n}@s{sd}" for sd in SEEDS] for n in VAR10}
rows = {}
for lab, ks in G10.items():
    r = {}
    for g, m in SEG3.items():
        v = [cvf(RES[k]) if m is None else seg(RES[k], m) for k in ks]
        r[g] = np.mean(v); r[g + " 범위"] = max(v) - min(v)
    rows[lab] = r
tab = pd.DataFrame(rows).T
for g in SEG3: tab[g + " Δ"] = tab[g] - tab.loc["기존 (선형 리프)", g]
print(tab[[c for g in SEG3 for c in (g, g + " Δ", g + " 범위")]].round(4).to_string())

base10 = np.mean([ffold(k) for k in G10["기존 (선형 리프)"]], axis=0)
for n in VAR10:
    d = np.mean([ffold(k) for k in G10[n]], axis=0) - base10
    print(f"{n:13s}: 전체 fold Δ {np.round(d, 4)} | 개선 fold {(d < 0).sum()}/5")

In [16]:
## [E-11] baseline 진단: (1) 오차를 계 수준(level)과 계 안 모양(shape)으로 분해  (2) mol/l seen에서 오차가 큰 계
sys_lab = (X["salt"] + " | " + X["c units"] + " | " + X["solvent ratio type"] + " | "
           + X[SOLVENTS].apply(lambda r: "+".join(f"{k}:{v:.2f}" for k, v in r[r > 0].items()), axis=1)).to_numpy()

def decompose(o, mask):  ## level = 계별 평균 오차, shape = 계 안에서 평균을 뺀 오차
    e = (o - LY)[mask]
    lvl = pd.Series(e).groupby(SYS[mask]).transform("mean").to_numpy()
    return {"RMSLE": np.sqrt((e ** 2).mean()), "level 몫": (lvl ** 2).sum() / (e ** 2).sum(),
            "shape RMSE": np.sqrt(((e - lvl) ** 2).mean())}

for n in ["R_base", "B1_unit", "W0.5@s0"]:   ## W0.5@s0: 지금 쓰는 트리 모델 (비교 기준)
    print(f"{n:9s}", {g: {k: round(float(v), 3) for k, v in decompose(RES[n], m).items()}
                     for g, m in [("mol/kg", IS_KG), ("mol/l", ~IS_KG)]})

S = SEG["mol/l seen"]; e = RES["B1_unit"] - LY
d = pd.DataFrame({"sys": sys_lab[S], "e": e[S], "T": T[S], "c": C[S]})
top = d.groupby("sys").agg(n=("e", "size"), SE=("e", lambda x: (x ** 2).sum()), bias=("e", "mean"), shape_sd=("e", "std"),
                           T_min=("T", "min"), T_max=("T", "max"), c_min=("c", "min"), c_max=("c", "max"))
top["SE 몫"] = top["SE"] / top["SE"].sum()
with pd.option_context("display.max_colwidth", 60, "display.width", 250):
    print("\n== B1_unit: mol/l seen 오차 상위 계 ==")
    print(top.sort_values("SE", ascending=False).head(10).round(3).to_string())

R_base    {'mol/kg': {'RMSLE': 0.254, 'level 몫': 0.46, 'shape RMSE': 0.187}, 'mol/l': {'RMSLE': 0.463, 'level 몫': 0.642, 'shape RMSE': 0.277}}
B1_unit   {'mol/kg': {'RMSLE': 0.302, 'level 몫': 0.293, 'shape RMSE': 0.254}, 'mol/l': {'RMSLE': 0.488, 'level 몫': 0.542, 'shape RMSE': 0.33}}
W0.5@s0   {'mol/kg': {'RMSLE': 0.149, 'level 몫': 0.154, 'shape RMSE': 0.137}, 'mol/l': {'RMSLE': 0.213, 'level 몫': 0.513, 'shape RMSE': 0.149}}

== B1_unit: mol/l seen 오차 상위 계 ==
                                                     n      SE   bias  shape_sd    T_min    T_max  c_min  c_max   SE 몫
sys                                                                                                                   
LiBPFPB | mol/l | mol | DME:1.00                    74  98.528 -1.057     0.466  212.970  331.237  0.200  0.600  0.282
LiBPFPB | mol/l | mol | PC:1.00                    154  62.701  0.190     0.611  211.988  477.423  0.077  2.570  0.179
LiBOB | mol/l | mol | DME:1.00                     213  38.

In [17]:
## [E-12] B5: 온도 전용 baseline (농도 항 없음) — 온도 의존성만 물리식이, 계마다 다른 농도 봉우리는 트리가 맡음
BASIS5 = pd.concat([pd.DataFrame({"lnetaT": lnetaT, "invT": invT_, "invT150": 1000 / (T - 150), "lnepsT": np.log(epsT)}),
                    pd.get_dummies(X[["salt"]], dtype=float).reset_index(drop=True)], axis=1)
B5M = BASIS5.to_numpy(float)

def fit_baseline_M(M, fi, w, per_unit=True):  ## E-10 fit_baseline과 같고, 기저 행렬 M만 인자로
    parts = [IS_KG, ~IS_KG] if per_unit else [np.ones(len(y), bool)]
    models = []
    for gm in parts:
        sel = gm[fi]
        m = mkp(StandardScaler(), Ridge(alpha=1.0)).fit(M[fi][sel], LNK[fi][sel], ridge__sample_weight=w[sel])
        models.append((gm, m))
    def predict(idx):
        out = np.zeros(len(idx))
        for gm, m in models:
            s = gm[idx]
            if s.any(): out[s] = m.predict(M[idx][s])
        return np.log1p(np.exp(out))
    return predict

def oof_base_M(M, per_unit):
    ob = np.zeros(len(y))
    for fi, vi in SPLITS:
        ob[vi] = fit_baseline_M(M, fi, fold_w(fi), per_unit)(vi)
    return ob

## 1) 진단: 온도 방향 모양 오차 (같은 조성 = 같은 c 안에서 평균을 뺀 오차, 온도 3점 이상인 조성만)
FORM = pd.factorize(groups)[0]
def decompose_T(o, mask):
    e = (o - LY)[mask]; f = FORM[mask]
    lvl = pd.Series(e).groupby(f).transform("mean").to_numpy()
    multi = pd.Series(f).map(pd.Series(f).value_counts()).to_numpy() >= 3
    return {"RMSLE": np.sqrt((e ** 2).mean()), "T-shape RMSE": np.sqrt(((e - lvl)[multi] ** 2).mean()), "3점 이상 행 비율": multi.mean()}

for n, pu in [("B5_glob", False), ("B5_unit", True)]:
    if n not in RES: RES[n] = oof_base_M(B5M, pu)
for n in ["B1_unit", "B5_glob", "B5_unit", "W0.5@s0"]:
    print(f"{n:8s}", {g: {k: round(float(v), 3) for k, v in decompose_T(RES[n], m).items()}
                      for g, m in [("mol/kg", IS_KG), ("mol/l", ~IS_KG)]})
print("↑ 확인 지점: B5_unit의 mol/l T-shape RMSE가 W0.5@s0(트리)에 가까우면 온도 의존성을 baseline이 맡을 수 있음\n")

## 2) 잔차 학습: B5 단위별 baseline + 트리 (상수 리프 / 선형 리프) × seed 2개
def oof_resid5(F, seed, linear_leaf):
    A = F.to_numpy(float); o = np.zeros(len(y))
    for fi, vi in SPLITS:
        w = fold_w(fi); bpred = fit_baseline_M(B5M, fi, w, True)
        bf, bv = bpred(fi), bpred(vi)
        m = LGBMRegressor(**(LGBP | {"random_state": seed, "linear_tree": linear_leaf})).fit(A[fi], LY[fi] - bf, sample_weight=w)
        o[vi] = np.clip(bv + m.predict(A[vi]), 0, None)
    return o

VAR12 = {"R5_gbdt": False, "R5_lin": True}
for n, ll in VAR12.items():
    for sd in SEEDS:
        key = f"{n}@s{sd}"
        if key not in RES:
            t0 = time.time(); RES[key] = oof_resid5(FB, sd, ll); print(f"{key:12s} {time.time() - t0:4.0f}s")

## 3) 구간별 seed 평균 / 범위 / 기존(E-7 α=0.5, 선형 리프) 대비 Δ
G12 = {"기존 (선형 리프)": [f"W0.5@s{sd}" for sd in SEEDS], "R_gbdt (E-9)": [f"R_gbdt@s{sd}" for sd in SEEDS]} \
      | {n: [f"{n}@s{sd}" for sd in SEEDS] for n in VAR12}
rows = {}
for lab, ks in G12.items():
    r = {}
    for g, m in SEG3.items():
        v = [cvf(RES[k]) if m is None else seg(RES[k], m) for k in ks]
        r[g] = np.mean(v); r[g + " 범위"] = max(v) - min(v)
    rows[lab] = r
tab = pd.DataFrame(rows).T
for g in SEG3: tab[g + " Δ"] = tab[g] - tab.loc["기존 (선형 리프)", g]
print(tab[[c for g in SEG3 for c in (g, g + " Δ", g + " 범위")]].round(4).to_string())

base12 = np.mean([ffold(k) for k in G12["기존 (선형 리프)"]], axis=0)
for n in VAR12:
    d = np.mean([ffold(k) for k in G12[n]], axis=0) - base12
    print(f"{n:8s}: 전체 fold Δ {np.round(d, 4)} | 개선 fold {(d < 0).sum()}/5")

B1_unit  {'mol/kg': {'RMSLE': 0.302, 'T-shape RMSE': 0.192, '3점 이상 행 비율': 0.516}, 'mol/l': {'RMSLE': 0.488, 'T-shape RMSE': 0.251, '3점 이상 행 비율': 0.915}}
B5_glob  {'mol/kg': {'RMSLE': 0.317, 'T-shape RMSE': 0.144, '3점 이상 행 비율': 0.516}, 'mol/l': {'RMSLE': 0.55, 'T-shape RMSE': 0.235, '3점 이상 행 비율': 0.915}}
B5_unit  {'mol/kg': {'RMSLE': 0.289, 'T-shape RMSE': 0.135, '3점 이상 행 비율': 0.516}, 'mol/l': {'RMSLE': 0.549, 'T-shape RMSE': 0.234, '3점 이상 행 비율': 0.915}}
W0.5@s0  {'mol/kg': {'RMSLE': 0.149, 'T-shape RMSE': 0.05, '3점 이상 행 비율': 0.516}, 'mol/l': {'RMSLE': 0.213, 'T-shape RMSE': 0.116, '3점 이상 행 비율': 0.915}}
↑ 확인 지점: B5_unit의 mol/l T-shape RMSE가 W0.5@s0(트리)에 가까우면 온도 의존성을 baseline이 맡을 수 있음

R5_gbdt@s0     71s


KeyboardInterrupt: 

In [18]:
## [E-13] 농도 의존 활성화에너지를 baseline 식에 넣기: ln k = A_salt − (Ea0 + Ea1·c)/RT → 1/T × c 항
cT = C * invT_                                                          ## c × (1000/T)
SALT_OH = pd.get_dummies(X["salt"], dtype=float).to_numpy()
B6a = np.c_[B5M, cT]                                                    ## 공통 Ea 기울기
B6b = np.c_[B5M, cT, SALT_OH * cT[:, None]]                             ## 염마다 다른 Ea 기울기
for n, M in [("B6a_unit", B6a), ("B6b_unit", B6b)]:
    if n not in RES: RES[n] = oof_base_M(M, True)
for n in ["R_base", "B5_unit", "B6a_unit", "B6b_unit", "W0.5@s0"]:
    print(f"{n:9s}", {g: {k: round(float(v), 3) for k, v in decompose_T(RES[n], m).items()}
                      for g, m in [("mol/kg", IS_KG), ("mol/l", ~IS_KG)]})
print("↑ 기준: 트리(W0.5@s0)의 mol/l T-shape 0.116, B5_unit 0.234")

R_base    {'mol/kg': {'RMSLE': 0.254, 'T-shape RMSE': 0.112, '3점 이상 행 비율': 0.516}, 'mol/l': {'RMSLE': 0.463, 'T-shape RMSE': 0.179, '3점 이상 행 비율': 0.915}}
B5_unit   {'mol/kg': {'RMSLE': 0.289, 'T-shape RMSE': 0.135, '3점 이상 행 비율': 0.516}, 'mol/l': {'RMSLE': 0.549, 'T-shape RMSE': 0.234, '3점 이상 행 비율': 0.915}}
B6a_unit  {'mol/kg': {'RMSLE': 0.299, 'T-shape RMSE': 0.146, '3점 이상 행 비율': 0.516}, 'mol/l': {'RMSLE': 0.529, 'T-shape RMSE': 0.242, '3점 이상 행 비율': 0.915}}
B6b_unit  {'mol/kg': {'RMSLE': 0.287, 'T-shape RMSE': 0.145, '3점 이상 행 비율': 0.516}, 'mol/l': {'RMSLE': 0.506, 'T-shape RMSE': 0.245, '3점 이상 행 비율': 0.915}}
W0.5@s0   {'mol/kg': {'RMSLE': 0.149, 'T-shape RMSE': 0.05, '3점 이상 행 비율': 0.516}, 'mol/l': {'RMSLE': 0.213, 'T-shape RMSE': 0.116, '3점 이상 행 비율': 0.915}}
↑ 기준: 트리(W0.5@s0)의 mol/l T-shape 0.116, B5_unit 0.234


In [19]:
## [E-14] B4-lite 1단계: 조성(같은 염·c·용매)마다 ln(1+k)를 온도 식으로 맞췄을 때의 잔차 = 이 구조의 바닥
invT150_ = 1000 / (T - 150)
ROWS = pd.Series(np.arange(len(y)))

def fit_per_comp(basis_fn, min_pts):
    res = np.full(len(y), np.nan); params = {}
    for f, idx in ROWS.groupby(FORM):
        idx = idx.to_numpy()
        if len(np.unique(np.round(T[idx], 2))) < min_pts: continue        ## 서로 다른 온도 점 수
        Bm = basis_fn(idx)
        coef, *_ = np.linalg.lstsq(Bm, LY[idx], rcond=None)
        res[idx] = LY[idx] - Bm @ coef; params[f] = coef
    return res, params

FORMS1 = {
    "Arrhenius  a+b/T":        (lambda i: np.c_[np.ones(len(i)), invT_[i]], 3),
    "VTF(T0=150) a+b/(T-150)": (lambda i: np.c_[np.ones(len(i)), invT150_[i]], 3),
    "곡률  a+b/T+d/T²":        (lambda i: np.c_[np.ones(len(i)), invT_[i], invT_[i] ** 2], 4),
}

def tree_tshape(mask):  ## 트리(W0.5@s0)의 T-shape: 같은 조성 안에서 평균을 뺀 OOF 오차, 같은 행에서
    e = (RES["W0.5@s0"] - LY)[mask]
    lvl = pd.Series(e).groupby(FORM[mask]).transform("mean").to_numpy()
    return np.sqrt(((e - lvl) ** 2).mean())

rows = {}
for name, (fn, mp) in FORMS1.items():
    res, params = fit_per_comp(fn, mp)
    cov = ~np.isnan(res)
    for u, um in [("mol/kg", IS_KG), ("mol/l", ~IS_KG)]:
        m = cov & um
        rows[(name, u)] = {"조성 수": len({f for f in params if um[FORM == f][0]}), "덮는 행 비율": m.sum() / um.sum(),
                           "1단계 잔차 RMSE": np.sqrt(np.mean(res[m] ** 2)), "잔차 95%": np.quantile(np.abs(res[m]), 0.95),
                           "트리 T-shape (같은 행)": tree_tshape(m)}
print(pd.DataFrame(rows).T.round(4).to_string())

## 물리 점검: Arrhenius 기울기로 본 겉보기 활성화에너지 (ln(1+k) 기준 근사, kJ/mol)
_, pA = fit_per_comp(*FORMS1["Arrhenius  a+b/T"])
Ea = pd.Series({f: -c[1] * 8.314 for f, c in pA.items()})
unit_of = pd.Series(np.where(IS_KG, "mol/kg", "mol/l")).groupby(FORM).first()
print("\n== 겉보기 Ea 분포 (kJ/mol) ==")
print(Ea.groupby(unit_of.loc[Ea.index].to_numpy()).describe(percentiles=[.05, .25, .5, .75, .95]).round(1).to_string())

                                 조성 수  덮는 행 비율  1단계 잔차 RMSE  잔차 95%  트리 T-shape (같은 행)
Arrhenius  a+b/T        mol/kg  562.0   0.5157       0.1269  0.3188             0.0501
                        mol/l   181.0   0.9129       0.1092  0.2352             0.1159
VTF(T0=150) a+b/(T-150) mol/kg  562.0   0.5157       0.2264  0.5556             0.0501
                        mol/l   181.0   0.9129       0.1628  0.3761             0.1159
곡률  a+b/T+d/T²          mol/kg  477.0   0.4857       0.0453  0.1098             0.0474
                        mol/l   173.0   0.9025       0.0620  0.1263             0.1164

== 겉보기 Ea 분포 (kJ/mol) ==
        count  mean  std  min   5%   25%   50%   75%   95%   max
mol/kg  562.0  11.7  3.1  0.4  6.5  10.0  11.6  13.4  16.5  24.7
mol/l   181.0  12.1  3.8  0.1  5.6  10.5  12.7  14.4  17.1  28.1


In [20]:
## [E-15] B4-lite 2단계: 조성별 (a, b, d) [ln(1+k) = a + b·x + d·x², x = 1000/T − 3.5] → 온도와 무관한 조성 feature로 GBDT 예측
X0 = 3.5                                                                ## 기준 ≈ 286 K: a = 그 온도의 ln(1+k), b = 기울기, d = 곡률
XC = invT_ - X0
CF = pd.concat([ONEHOT, RAW.drop(columns="T"), M7B, SD,
                pd.DataFrame({"lnc": lnc_, "sqc": np.sqrt(C)}
                             | {f"Bmix_{b}": xb @ BAND for b, xb in enumerate(XB)}            ## 혼합 점도 활성화에너지
                             | {f"dEps_{b}": xb @ (EPS25 - NN) for b, xb in enumerate(XB)})], ## 유전율 온도 의존 크기
               axis=1)                                                  ## 모두 같은 조성 안에서 상수
CFM = CF.to_numpy(float)
P2 = dict(n_estimators=600, learning_rate=0.03, num_leaves=7, min_child_samples=10, subsample=0.8, subsample_freq=1,
          colsample_bytree=0.6, reg_lambda=5.0, n_jobs=-1, verbose=-1)

def comp_params(idx):  ## idx 행들 안에서, 서로 다른 온도 4점 이상인 조성의 (대표 행, a, b, d)
    out = []
    for f, ii in pd.Series(idx).groupby(FORM[idx]):
        ii = ii.to_numpy()
        if len(np.unique(np.round(T[ii], 2))) < 4: continue
        Bm = np.c_[np.ones(len(ii)), XC[ii], XC[ii] ** 2]
        coef, *_ = np.linalg.lstsq(Bm, LY[ii], rcond=None)
        out.append((ii[0], *coef))
    return np.array(out)

UNPHYS = {}
def oof_stage2(seed):
    o = np.zeros(len(y)); slope = np.zeros(len(y)); n_comp = []
    for fi, vi in SPLITS:
        cp = comp_params(fi); rep = cp[:, 0].astype(int); n_comp.append(len(cp))
        a, b, d = [LGBMRegressor(**(P2 | {"random_state": seed})).fit(CFM[rep], cp[:, 1 + j]).predict(CFM[vi]) for j in range(3)]
        o[vi] = np.clip(a + b * XC[vi] + d * XC[vi] ** 2, 0, None)
        slope[vi] = b + 2 * d * XC[vi]                                  ## dln(1+k)/dx > 0 이면 T↑에 k↓ (물리 위반)
    UNPHYS[seed] = slope > 0
    print(f"seed {seed}: fold별 학습 조성 수 {n_comp}")
    return o

for sd in SEEDS:
    if f"S2@s{sd}" not in RES: RES[f"S2@s{sd}"] = oof_stage2(sd)
    RES[f"S2+W@s{sd}"] = (RES[f"S2@s{sd}"] + RES[f"W0.5@s{sd}"]) / 2   ## 기존 트리와 log 공간 평균

## 구간별 seed 평균 / 범위 / 기존(W0.5) 대비 Δ
G15 = {"기존 (W0.5)": [f"W0.5@s{sd}" for sd in SEEDS], "2단계 단독": [f"S2@s{sd}" for sd in SEEDS],
       "2단계 + 기존 평균": [f"S2+W@s{sd}" for sd in SEEDS]}
rows = {}
for lab, ks in G15.items():
    r = {}
    for g, m in SEG3.items():
        v = [cvf(RES[k]) if m is None else seg(RES[k], m) for k in ks]
        r[g] = np.mean(v); r[g + " 범위"] = max(v) - min(v)
    rows[lab] = r
tab = pd.DataFrame(rows).T
for g in SEG3: tab[g + " Δ"] = tab[g] - tab.loc["기존 (W0.5)", g]
print(tab[[c for g in SEG3 for c in (g, g + " Δ", g + " 범위")]].round(4).to_string())

for lab in ["2단계 단독", "2단계 + 기존 평균"]:
    for g in ["mol/l seen", "mol/l unseen"]:
        m = SEG[g]
        ff = lambda k: np.array([np.sqrt(((RES[k][vi] - LY[vi])[m[vi]] ** 2).mean()) for _, vi in SPLITS])
        d_ = np.mean([ff(k) for k in G15[lab]], axis=0) - np.mean([ff(k) for k in G15["기존 (W0.5)"]], axis=0)
        print(f"{lab:10s} {g:12s}: fold Δ {np.round(d_, 3)} | 개선 fold {(d_ < 0).sum()}/5")
print("\n물리 위반 행 비율 (T↑인데 k↓), seed 0:", {g: round(float(UNPHYS[0][m].mean()), 3) for g, m in SEG.items()})

seed 0: fold별 학습 조성 수 [520, 520, 520, 520, 520]
seed 1: fold별 학습 조성 수 [520, 520, 520, 520, 520]
              전체 cv  전체 cv Δ  전체 cv 범위  mol/kg  mol/kg Δ  mol/kg 범위  mol/l seen  mol/l seen Δ  mol/l seen 범위  mol/l unseen  mol/l unseen Δ  mol/l unseen 범위
기존 (W0.5)    0.1627   0.0000    0.0003  0.1488    0.0000     0.0001      0.2058        0.0000         0.0026        0.2200          0.0000           0.0012
2단계 단독       0.2893   0.1266    0.0007  0.2172    0.0684     0.0000      0.5592        0.3533         0.0023        0.3778          0.1578           0.0060
2단계 + 기존 평균  0.2028   0.0401    0.0001  0.1659    0.0171     0.0001      0.3423        0.1365         0.0034        0.2715          0.0515           0.0029
2단계 단독     mol/l seen  : fold Δ [0.274 0.496 0.402 0.324 0.136] | 개선 fold 0/5
2단계 단독     mol/l unseen: fold Δ [0.078 0.086 0.197 0.101 0.275] | 개선 fold 0/5
2단계 + 기존 평균 mol/l seen  : fold Δ [0.04  0.22  0.172 0.138 0.034] | 개선 fold 0/5
2단계 + 기존 평균 mol/l unseen: fold Δ [0.027 0.02 

In [21]:
## [E-16] 2단계 진단: 오차를 수준(조성 평균)과 온도 방향 모양으로 분해 — 모양이 맞았다면 "모양 baseline + 트리" 조합이 가능
for n in ["W0.5@s0", "S2@s0", "S2@s1"]:
    print(f"{n:8s}", {g: {k: round(float(v), 3) for k, v in decompose_T(RES[n], m).items()}
                      for g, m in [("mol/kg", IS_KG), ("mol/l seen", SEG["mol/l seen"]), ("mol/l unseen", SEG["mol/l unseen"])]})

## 수준 오차: 조성별 평균 오차의 RMS (조성 하나를 한 번씩 셈)
for n in ["W0.5@s0", "S2@s0"]:
    for g in ["mol/l seen", "mol/l unseen"]:
        m = SEG[g]
        lvl = pd.Series((RES[n] - LY)[m]).groupby(FORM[m]).mean()
        print(f"{n:8s} {g:12s} 수준 오차 RMS (조성 단위): {np.sqrt((lvl ** 2).mean()):.3f} | 조성 수 {len(lvl)}")

W0.5@s0  {'mol/kg': {'RMSLE': 0.149, 'T-shape RMSE': 0.05, '3점 이상 행 비율': 0.516}, 'mol/l seen': {'RMSLE': 0.205, 'T-shape RMSE': 0.129, '3점 이상 행 비율': 0.84}, 'mol/l unseen': {'RMSLE': 0.221, 'T-shape RMSE': 0.105, '3점 이상 행 비율': 0.981}}
S2@s0    {'mol/kg': {'RMSLE': 0.217, 'T-shape RMSE': 0.105, '3점 이상 행 비율': 0.516}, 'mol/l seen': {'RMSLE': 0.558, 'T-shape RMSE': 0.252, '3점 이상 행 비율': 0.84}, 'mol/l unseen': {'RMSLE': 0.381, 'T-shape RMSE': 0.204, '3점 이상 행 비율': 0.981}}
S2@s1    {'mol/kg': {'RMSLE': 0.217, 'T-shape RMSE': 0.104, '3점 이상 행 비율': 0.516}, 'mol/l seen': {'RMSLE': 0.56, 'T-shape RMSE': 0.257, '3점 이상 행 비율': 0.84}, 'mol/l unseen': {'RMSLE': 0.375, 'T-shape RMSE': 0.204, '3점 이상 행 비율': 0.981}}
W0.5@s0  mol/l seen   수준 오차 RMS (조성 단위): 0.124 | 조성 수 212
W0.5@s0  mol/l unseen 수준 오차 RMS (조성 단위): 0.191 | 조성 수 144
S2@s0    mol/l seen   수준 오차 RMS (조성 단위): 0.370 | 조성 수 212
S2@s0    mol/l unseen 수준 오차 RMS (조성 단위): 0.306 | 조성 수 144


In [22]:
## [E-18] 성분 벡터 기반 염–용매 쌍 효과 (matrix completion: Jirasek et al., Chem. Sci. 2022의 표 형식 버전)
## 쌍 효과_i = u_salt(i) · (Σ_j x_ij v_j)  — 염 벡터 u (13×k), 용매 벡터 v (38×k), x = 몰분율
SALTS_U = np.unique(salt)
S_IDX = pd.Series(np.arange(len(SALTS_U)), index=SALTS_U)[salt].to_numpy()
B9 = BASIS.to_numpy(float)                                              ## E-9 물리 baseline 기저 (+ 염·단위 절편)

def fit_pair(rows, w, k, lam=1.0, n_iter=25, seed=0):
    base = mkp(StandardScaler(), Ridge(alpha=1.0)).fit(B9[rows], LY[rows], ridge__sample_weight=w)
    r = LY[rows] - base.predict(B9[rows])                               ## backbone 잔차를 쌍 효과로 분해
    V = np.random.default_rng(seed).normal(0, 0.1, (len(SOLVENTS), k)); U = np.zeros((len(SALTS_U), k))
    Xm, s = XM[rows], S_IDX[rows]
    for _ in range(n_iter):                                             ## 교대 최소제곱 (ridge)
        Z = Xm @ V
        for si in range(len(SALTS_U)):
            m = s == si
            if m.any():
                Zw = Z[m] * w[m, None]
                U[si] = np.linalg.solve(Z[m].T @ Zw + lam * np.eye(k), Zw.T @ r[m])
        Fv = (Xm[:, :, None] * U[s][:, None, :]).reshape(len(rows), -1)  ## x_ij · u_s(i),k
        Fw = Fv * w[:, None]
        V = np.linalg.solve(Fv.T @ Fw + lam * np.eye(Fv.shape[1]), Fw.T @ r).reshape(len(SOLVENTS), k)
    return base, U, V

def pair_feats(idx, U, V):  ## [u ⊙ v_mix (k열), 쌍 효과 합 (1열)]
    P = U[S_IDX[idx]] * (XM[idx] @ V)
    return np.c_[P, P.sum(1)]

def pair_feats_oof(fi, w, k, seed):  ## 학습 행의 쌍 feature: 학습 fold 안에서 계 단위 5-fold → 자기 계를 보지 않은 벡터로 계산
    Pf = np.zeros((len(fi), k + 1))
    for a, b in GroupKFold(n_splits=5).split(fi, groups=SYS[fi]):
        _, U, V = fit_pair(fi[a], w[a] / w[a].mean(), k, seed=seed)
        Pf[b] = pair_feats(fi[b], U, V)
    return Pf

## 1) 선형 수준 진단: baseline 단독 vs baseline + 쌍 효과 (처음 보는 계로 옮겨 가는가)
lin = {}
for k in [1, 2, 3]:
    ob, op = np.zeros(len(y)), np.zeros(len(y))
    for fi, vi in SPLITS:
        base, U, V = fit_pair(fi, fold_w(fi), k)
        bv = base.predict(B9[vi]); ob[vi] = np.clip(bv, 0, None)
        op[vi] = np.clip(bv + pair_feats(vi, U, V)[:, -1], 0, None)
    lin["baseline 단독"] = {g: seg(ob, m) for g, m in SEG.items()}
    lin[f"baseline + 쌍 효과 (k={k})"] = {g: seg(op, m) for g, m in SEG.items()}
print(pd.DataFrame(lin).T.round(4).to_string())

## 2) 트리에 쌍 feature 추가 (k = 2, 3) × seed 2개
def oof_pair_tree(F, k, seed):
    A = F.to_numpy(float); o = np.zeros(len(y))
    for fi, vi in SPLITS:
        w = fold_w(fi)
        Pf = pair_feats_oof(fi, w, k, seed)                             ## 학습 행: 누수 없는 쌍 feature
        _, U, V = fit_pair(fi, w, k, seed=seed); Pv = pair_feats(vi, U, V)   ## 검증 행: 학습 fold 전체로 맞춘 벡터
        m = LGBMRegressor(**(LGBP | {"random_state": seed})).fit(np.c_[A[fi], Pf], LY[fi], sample_weight=w)
        o[vi] = np.clip(m.predict(np.c_[A[vi], Pv]), 0, None)
    return o

for k in [2, 3]:
    for sd in SEEDS:
        key = f"PAIR{k}@s{sd}"
        if key not in RES:
            t0 = time.time(); RES[key] = oof_pair_tree(FB, k, sd); print(f"{key:10s} {time.time() - t0:4.0f}s")

G18 = {"기존 (W0.5)": [f"W0.5@s{sd}" for sd in SEEDS]} | {f"+ 쌍 feature k={k}": [f"PAIR{k}@s{sd}" for sd in SEEDS] for k in [2, 3]}
rows = {}
for lab, ks in G18.items():
    r_ = {}
    for g, m in SEG3.items():
        v = [cvf(RES[kk]) if m is None else seg(RES[kk], m) for kk in ks]
        r_[g] = np.mean(v); r_[g + " 범위"] = max(v) - min(v)
    rows[lab] = r_
tab = pd.DataFrame(rows).T
for g in SEG3: tab[g + " Δ"] = tab[g] - tab.loc["기존 (W0.5)", g]
print(tab[[c for g in SEG3 for c in (g, g + " Δ", g + " 범위")]].round(4).to_string())
for k in [2, 3]:
    for g in ["mol/l seen", "mol/l unseen"]:
        m = SEG[g]
        ff = lambda kk: np.array([np.sqrt(((RES[kk][vi] - LY[vi])[m[vi]] ** 2).mean()) for _, vi in SPLITS])
        d_ = np.mean([ff(kk) for kk in G18[f"+ 쌍 feature k={k}"]], 0) - np.mean([ff(kk) for kk in G18["기존 (W0.5)"]], 0)
        print(f"k={k} {g:12s}: fold Δ {np.round(d_, 3)} | 개선 fold {(d_ < 0).sum()}/5")

## 3) 해석: 전체 데이터로 학습한 염 벡터 (k=2)
_, U_all, V_all = fit_pair(np.arange(len(y)), np.ones(len(y)), 2)
print("\n== 염 벡터 u (k=2) ==");  print(pd.DataFrame(U_all, index=SALTS_U, columns=["u1", "u2"]).round(3).to_string())
print("\n== 용매 벡터 v (k=2), 크기 상위 12 ==")
Vdf = pd.DataFrame(V_all, index=SOLVENTS, columns=["v1", "v2"]); Vdf["|v|"] = np.linalg.norm(V_all, axis=1)
print(Vdf.sort_values("|v|", ascending=False).head(12).round(3).to_string())

                       mol/kg  mol/l seen  mol/l unseen
baseline 단독            0.2543      0.5776        0.3333
baseline + 쌍 효과 (k=1)  0.2330      0.4594        0.3669
baseline + 쌍 효과 (k=2)  0.2254      0.4017        0.4120
baseline + 쌍 효과 (k=3)  0.2232      0.3905        0.3638
PAIR2@s0     94s
PAIR2@s1     80s
PAIR3@s0     85s
PAIR3@s1     85s
                  전체 cv  전체 cv Δ  전체 cv 범위  mol/kg  mol/kg Δ  mol/kg 범위  mol/l seen  mol/l seen Δ  mol/l seen 범위  mol/l unseen  mol/l unseen Δ  mol/l unseen 범위
기존 (W0.5)        0.1627   0.0000    0.0003  0.1488    0.0000     0.0001      0.2058        0.0000         0.0026        0.2200          0.0000           0.0012
+ 쌍 feature k=2  0.1703   0.0076    0.0015  0.1566    0.0078     0.0005      0.2136        0.0077         0.0104        0.2285          0.0085           0.0000
+ 쌍 feature k=3  0.1786   0.0159    0.0000  0.1623    0.0135     0.0002      0.2401        0.0342         0.0162        0.2316          0.0116           0.0049
k=2 mol/l se

In [23]:
## [E-19] 분율 합 지문: EC:DMC:DEC 1 m 이상 계는 용매 분율 합이 0.99 → 이 출처의 이상을 모델이 알아볼 수 있는가
FSUM_DEV = X[SOLVENTS].sum(axis=1).to_numpy() - 1.0
FP = np.abs(FSUM_DEV) > 0.005
fold_of = np.zeros(len(y), int)
for i, (_, vi) in enumerate(SPLITS): fold_of[vi] = i
print("== 지문 행(분율 합 ≠ 1)의 fold 배치 ==")
print(pd.crosstab([X["salt"][FP], X["c units"][FP]], fold_of[FP]).to_string())
print("지문 행의 k 범위:", np.round(np.expm1(LY[FP & IS_KG]).min(), 3), "~", np.round(np.expm1(LY[FP & IS_KG]).max(), 3))

FB_FP = pd.concat([FB, pd.DataFrame({"fsum_dev": FSUM_DEV})], axis=1)
for sd in SEEDS:
    key = f"FP@s{sd}"
    if key not in RES:
        t0 = time.time(); RES[key] = oof_w(FB_FP, 0.5, sd); print(f"{key:8s} {time.time() - t0:4.0f}s")

SEG19 = {"전체 cv": None, "mol/kg 지문 행": IS_KG & FP, "mol/kg 나머지": IS_KG & ~FP,
         "mol/l seen": SEG["mol/l seen"], "mol/l unseen": SEG["mol/l unseen"]}
G19 = {"기존 (W0.5)": [f"W0.5@s{sd}" for sd in SEEDS], "+ 분율 합 지문": [f"FP@s{sd}" for sd in SEEDS]}
rows = {}
for lab, ks in G19.items():
    r_ = {}
    for g, m in SEG19.items():
        v = [cvf(RES[kk]) if m is None else seg(RES[kk], m) for kk in ks]
        r_[g] = np.mean(v); r_[g + " 범위"] = max(v) - min(v)
    rows[lab] = r_
tab = pd.DataFrame(rows).T
for g in SEG19: tab[g + " Δ"] = tab[g] - tab.loc["기존 (W0.5)", g]
print(tab[[c for g in SEG19 for c in (g, g + " Δ", g + " 범위")]].round(4).to_string())
d_ = np.mean([ffold(k) for k in G19["+ 분율 합 지문"]], 0) - np.mean([ffold(k) for k in G19["기존 (W0.5)"]], 0)
print("전체 cv fold Δ:", np.round(d_, 4), f"| 개선 fold {(d_ < 0).sum()}/5")

== 지문 행(분율 합 ≠ 1)의 fold 배치 ==
col_0              0  1   2  3
salt     c units              
LiBF4    mol/kg    0  0  11  0
LiCF3SO3 mol/l     1  1   0  1
LiClO4   mol/l     1  1   1  0
LiPF6    mol/kg   12  0   0  0
LiTFSI   mol/l     1  0   0  1
지문 행의 k 범위: 0.025 ~ 0.161
FP@s0      83s
FP@s1      60s
            전체 cv  전체 cv Δ  전체 cv 범위  mol/kg 지문 행  mol/kg 지문 행 Δ  mol/kg 지문 행 범위  mol/kg 나머지  mol/kg 나머지 Δ  mol/kg 나머지 범위  mol/l seen  mol/l seen Δ  mol/l seen 범위  mol/l unseen  mol/l unseen Δ  mol/l unseen 범위
기존 (W0.5)  0.1627    0.000    0.0003       1.1877         0.0000          0.0034      0.1355        0.0000         0.0000      0.2058         0.000         0.0026        0.2200          0.0000           0.0012
+ 분율 합 지문  0.1607   -0.002    0.0025       1.0965        -0.0912          0.0228      0.1357        0.0002         0.0005      0.1968        -0.009         0.0260        0.2214          0.0014           0.0011
전체 cv fold Δ: [-0.0072 -0.0024  0.0014 -0.0021  0.0004] | 개선 fold 3